# Book Recommendation Engine using KNN 📚

Reproducible cosine-nearest-neighbor recommender using the Book-Crossings ratings dataset. The function returns five title/distance pairs.

In [ ]:
from pathlib import Path
from urllib.request import urlretrieve
import zipfile
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix
from sklearn.neighbors import NearestNeighbors

archive = Path("book-crossings.zip")
if not archive.exists():
    urlretrieve("https://cdn.freecodecamp.org/project-data/books/book-crossings.zip", archive)
if not Path("BX-Books.csv").exists():
    with zipfile.ZipFile(archive) as z:
        z.extractall(".")

df_books = pd.read_csv("BX-Books.csv", encoding="ISO-8859-1", sep=";", header=0,
    names=["isbn", "title", "author"], usecols=["isbn", "title", "author"],
    dtype={"isbn":str, "title":str, "author":str})
df_ratings = pd.read_csv("BX-Book-Ratings.csv", encoding="ISO-8859-1", sep=";", header=0,
    names=["user", "isbn", "rating"], usecols=["user", "isbn", "rating"],
    dtype={"user":"int32", "isbn":str, "rating":"float32"})
print("Books:",len(df_books),"Ratings:",len(df_ratings))


## Filter sparse users and books
Retain users who have at least 200 ratings and ISBNs with at least 100 ratings in the original dataset. Join ratings with readable book titles.

In [ ]:
user_counts = df_ratings["user"].value_counts()
book_counts = df_ratings["isbn"].value_counts()
ratings = df_ratings.loc[
    df_ratings["user"].isin(user_counts[user_counts >= 200].index)
    & df_ratings["isbn"].isin(book_counts[book_counts >= 100].index)
].merge(df_books[["isbn","title"]], on="isbn", how="inner")
book_user = ratings.pivot_table(index="title", columns="user", values="rating",
                                aggfunc="mean", fill_value=0)
sparse_matrix = csr_matrix(book_user.to_numpy(dtype=np.float32))
knn = NearestNeighbors(metric="cosine", algorithm="brute")
knn.fit(sparse_matrix)
print("Book x user matrix:",book_user.shape)


## get_recommends(book)
Five nearest books with cosine distances. The output ordering is reversed (farthest to closest among the selected five) to match freeCodeCamp's sample output.

In [ ]:
def get_recommends(book=""):
    if book not in book_user.index:
        raise ValueError(f"Book not found among filtered titles: {book}")
    index = book_user.index.get_loc(book)
    distances, indices = knn.kneighbors(sparse_matrix[index],
                                         n_neighbors=min(6,len(book_user)))
    matches = [
        [str(book_user.index[i]), float(distance)]
        for i, distance in zip(indices[0], distances[0])
        if i != index
    ]
    return [book, matches[:5][::-1]]

sample_title = "Where the Heart Is (Oprah's Book Club (Paperback))"
print(get_recommends(sample_title))


## Official-style verification
FreeCodeCamp tests the two farthest of the five nearest recommendations and checks their expected distance ranges.

In [ ]:
recommends = get_recommends(sample_title)
expected_titles = {"I'll Be Seeing You", "The Weight of Water",
                   "The Surgeon", "I Know This Much Is True"}
expected_distances = [.80,.77]
passed = (recommends[0] == sample_title and len(recommends[1]) == 5)
for i in range(2):
    passed = passed and recommends[1][i][0] in expected_titles
    passed = passed and abs(recommends[1][i][1] - expected_distances[i]) < .05
print("PASS" if passed else "NOT YET PASSING — inspect filtering and ordering")
